# 2D VAE on MNIST — Latent Space Demo (PyTorch)

**Colab:** Runtime → Run all. CPU is enough (~1–2 min). GPU optional.

**You will observe**
1. Digit clusters in a **2D latent space** (similar shapes **4** / **9** nearby)
2. **Linear interpolation** on a line **4 → 0**
3. **Reconstruction** quality (typically blurry)

Matches the whiteboard demo: encoder → μ,σ → reparameterization → decoder → reconstruction + KL.

## 0. Setup

In [ ]:
# Colab / local: ensure deps (Colab usually already has torch + torchvision)
import sys
import subprocess

def _ensure(pkgs):
    import importlib
    missing = []
    for p in pkgs:
        try:
            importlib.import_module(p)
        except ImportError:
            missing.append(p)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

_ensure(["torch", "torchvision", "matplotlib", "tqdm"])

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)

## 1. Model — 2D VAE (from scratch)

- Encoder outputs $\mu, \log\sigma^2$
- **Reparameterization:** $z = \mu + \sigma \odot \varepsilon$, $\varepsilon \sim \mathcal{N}(0,I)$ (so sampling is differentiable)
- Decoder reconstructs $\hat{x}$
- Loss = reconstruction (BCE) + KL (push latent toward $\mathcal{N}(0,I)$)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=2, hidden=400):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
        )
        self.fc_mu = nn.Linear(hidden, latent_dim)
        self.fc_logvar = nn.Linear(hidden, latent_dim)

    def forward(self, x):
        h = self.net(x)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder(nn.Module):
    def __init__(self, latent_dim=2, hidden=400):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, 28 * 28),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class VAE2D(nn.Module):
    def __init__(self, latent_dim=2, hidden=400):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = Encoder(latent_dim, hidden)
        self.decoder = Decoder(latent_dim, hidden)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps  # differentiable sampling

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

    @torch.no_grad()
    def encode_mu(self, x):
        mu, _ = self.encoder(x)
        return mu

    @torch.no_grad()
    def decode(self, z):
        return self.decoder(z)


def vae_loss(recon, x, mu, logvar, beta=1.0):
    bce = nn.functional.binary_cross_entropy(recon, x, reduction="sum") / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return bce + beta * kl, bce.detach(), kl.detach()


model = VAE2D(latent_dim=2, hidden=400).to(device)
print(model)

## 2. Data — MNIST

In [ ]:
BATCH = 128
tfm = transforms.ToTensor()
train_ds = datasets.MNIST("./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST("./data", train=False, download=True, transform=tfm)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)
print("train:", len(train_ds), "test:", len(test_ds))

## 3. Train

In [ ]:
EPOCHS = 12   # increase to 15–20 for clearer clusters
LR = 1e-3
BETA = 1.0

opt = torch.optim.Adam(model.parameters(), lr=LR)
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for x, _ in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}", leave=False):
        x = x.to(device)
        opt.zero_grad(set_to_none=True)
        recon, mu, logvar = model(x)
        loss, _, _ = vae_loss(recon, x, mu, logvar, beta=BETA)
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    train_loss = total / len(train_ds)

    model.eval()
    total = 0.0
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to(device)
            recon, mu, logvar = model(x)
            loss, _, _ = vae_loss(recon, x, mu, logvar, beta=BETA)
            total += loss.item() * x.size(0)
    test_loss = total / len(test_ds)
    history.append((train_loss, test_loss))
    print(f"epoch {epoch:02d}  train={train_loss:.2f}  test={test_loss:.2f}")

plt.figure(figsize=(5, 3))
plt.plot([h[0] for h in history], label="train")
plt.plot([h[1] for h in history], label="test")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("VAE loss")
plt.tight_layout(); plt.show()

## 4. Latent space scatter (all digits)

Each point = encoder $\mu(x)$ for one test image. Look for **4** and **9** overlapping / neighbouring.

In [ ]:
DIGIT_COLORS = {
    0: "#1f77b4", 1: "#ff7f0e", 2: "#2ca02c", 3: "#d62728", 4: "#9467bd",
    5: "#8c564b", 6: "#e377c2", 7: "#7f7f7f", 8: "#bcbd22", 9: "#17becf",
}

@torch.no_grad()
def collect_latents(max_points=8000):
    zs, ys = [], []
    n = 0
    for x, y in test_loader:
        x = x.to(device)
        mu = model.encode_mu(x).cpu().numpy()
        zs.append(mu); ys.append(y.numpy())
        n += x.size(0)
        if n >= max_points:
            break
    return np.concatenate(zs)[:max_points], np.concatenate(ys)[:max_points]

z, y = collect_latents()

fig, ax = plt.subplots(figsize=(7.5, 6.5))
for d in range(10):
    m = y == d
    ax.scatter(z[m, 0], z[m, 1], s=8, alpha=0.45, c=DIGIT_COLORS[d], label=str(d))
ax.set_title("MNIST 2D latent space (encoder μ)")
ax.set_xlabel("z₁"); ax.set_ylabel("z₂")
ax.legend(title="digit", markerscale=2, loc="best", fontsize=8)
ax.set_aspect("equal", adjustable="datalim"); ax.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()

## 5. Focus: digits 4 / 9 nearby + path **4 → 0**

Morphologically similar digits sit close in latent space. We draw the mean-to-mean line used for interpolation.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(z[:, 0], z[:, 1], s=4, c="#dddddd", alpha=0.35)
for d in (4, 9, 0):
    m = y == d
    ax.scatter(z[m, 0], z[m, 1], s=12, alpha=0.55, c=DIGIT_COLORS[d], label=f"digit {d}")
    cx, cy = z[m, 0].mean(), z[m, 1].mean()
    ax.scatter([cx], [cy], s=120, marker="*", c=DIGIT_COLORS[d], edgecolors="k", zorder=5)
    ax.annotate(str(d), (cx, cy), textcoords="offset points", xytext=(6, 6), fontsize=12)

c4 = z[y == 4].mean(axis=0)
c0 = z[y == 0].mean(axis=0)
ax.plot([c4[0], c0[0]], [c4[1], c0[1]], "k--", lw=1.5, label="4 → 0 mean path")
ax.set_title("Latent neighbourhood: similar shapes (4/9) & 4→0 path")
ax.set_xlabel("z₁"); ax.set_ylabel("z₂")
ax.legend(loc="best"); ax.set_aspect("equal", adjustable="datalim"); ax.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()

## 6. Linear interpolation: **4 → 0**

Sample points on the straight line between mean-$\mu$ of digit 4 and digit 0, then decode.

In [ ]:
@torch.no_grad()
def mean_latent(digit, n=200):
    xs = []
    for img, label in test_ds:
        if label == digit:
            xs.append(img)
            if len(xs) >= n:
                break
    batch = torch.stack(xs).to(device)
    return model.encode_mu(batch).mean(0)

z4 = mean_latent(4)
z0 = mean_latent(0)
STEPS = 12
alphas = torch.linspace(0, 1, STEPS, device=device)
zs = torch.stack([(1 - a) * z4 + a * z0 for a in alphas])
imgs = model.decode(zs).cpu().numpy()

fig, axes = plt.subplots(1, STEPS, figsize=(1.2 * STEPS, 1.8))
for i, ax in enumerate(axes):
    ax.imshow(imgs[i, 0], cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
    ax.set_title("4" if i == 0 else ("0" if i == STEPS - 1 else "→"), fontsize=10)
fig.suptitle("Linear interpolation in latent space: 4 → 0", y=1.08, fontsize=12)
plt.tight_layout(); plt.show()

## 7. Reconstruction quality — often blurry

Top: original MNIST. Bottom: VAE reconstruction. Soft edges are expected for a standard VAE.

In [ ]:
chosen = {}
for img, label in test_ds:
    if label not in chosen:
        chosen[label] = img
    if len(chosen) == 10:
        break

labels = list(range(10))
xs = torch.stack([chosen[d] for d in labels]).to(device)
with torch.no_grad():
    recons, _, _ = model(xs)
xs_np, rc_np = xs.cpu().numpy(), recons.cpu().numpy()

fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
for i, d in enumerate(labels):
    axes[0, i].imshow(xs_np[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(str(d), fontsize=9); axes[0, i].axis("off")
    axes[1, i].imshow(rc_np[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("original", fontsize=9)
axes[1, 0].set_ylabel("recon\n(blurry)", fontsize=8)
fig.suptitle("VAE reconstruction quality on MNIST", fontsize=12)
plt.tight_layout(); plt.show()

## Takeaways (for PPT / report)

| Observation | What it means |
|-------------|----------------|
| Digit clusters in 2D | Encoder learns a continuous latent map |
| **4** near **9** | Similar shapes → nearby latent codes |
| Smooth **4 → 0** walk | Linear interpolation in $z$ yields gradual morphs |
| Blurry recon | Classic VAE trade-off (ELBO / averaging) vs GANs / diffusion |

Optional: `torch.save(model.state_dict(), "vae2d_mnist.pt")` to keep the weights.

In [ ]:
# Optional: save weights in Colab
# torch.save({"model": model.state_dict(), "latent_dim": 2}, "vae2d_mnist.pt")
# from google.colab import files; files.download("vae2d_mnist.pt")
print("Done. Scroll up for latent scatter, 4→0 interpolation, and reconstructions.")